# Labeling Script also Providing Defnitions and Examples  

## TOC 

- [Setup](#setup)
- [Load, Partition, and Sample Tweets/Batches for Labeling](#load_tweets)
- [Define the different prompts](#define_prompts)
- [Run the Labeling Tasks (Single)](#run_labeling_single)
- [Run the Labeling Tasks (Single) + Confidence](#run_labeling_single_conf)
- [Run the Labeling Tasks (Batch)](#run_labeling_batch)
- [Run the Labeling Tasks (Batch) + Confidence](#run_labeling_batch_confidence)

#### Open ToDo's
- Clean tweets from nbsps etc.  
- rewrite data frame creation and data collection into separate functions  

- Update Collection prompt s.t. there are now unneccessary ' after splitting
 --> batch conf still contains some cases without proper scores

Update to previous versions: 

-separation of task by system message (prompt and def) and user message (tweet)

In [1]:
# General Setup and Git
# cd to path of current script
import os
import sys
os.chdir(sys.path[0])

! git pull

Already up to date.


In [2]:
# Git codes
## see git status
! git status # see git status

# Ask the user for a commit message
commit_message = input("Enter commit message: ")

## Git add the current stuff to commit and set commit message
! git add .
! git commit -m "$commit_message"

## Push commits to remote
! git push

On branch master
Your branch is up to date with 'origin/master'.

Changes not staged for commit:
  (use "git add/rm <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	deleted:    .ipynb_checkpoints/3c_GPT_Label_Certainty-checkpoint.ipynb
	modified:   .jupyter/desktop-workspaces/default-37a8.jupyterlab-workspace
	modified:   1a_GPT_Labeling.ipynb
	deleted:    3c_GPT_Label_Certainty.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.ipynb_checkpoints/1d_Separate_Labeling_Definition_FewShot_refined_sep-checkpoint.ipynb
	.ipynb_checkpoints/2c_GPT_Label_Certainty-checkpoint.ipynb
	1d_Separate_Labeling_Definition_FewShot_refined_sep.ipynb
	2c_GPT_Label_Certainty.ipynb

no changes added to commit (use "git add" and/or "git commit -a")


Enter commit message:  separate Labeling


[master 09d0bda] separate Labeling
 6 files changed, 7724 insertions(+), 3 deletions(-)
 create mode 100644 .ipynb_checkpoints/1d_Separate_Labeling_Definition_FewShot_refined_sep-checkpoint.ipynb
 rename .ipynb_checkpoints/{3c_GPT_Label_Certainty-checkpoint.ipynb => 2c_GPT_Label_Certainty-checkpoint.ipynb} (100%)
 rewrite .jupyter/desktop-workspaces/default-37a8.jupyterlab-workspace (89%)
 create mode 100644 1d_Separate_Labeling_Definition_FewShot_refined_sep.ipynb
 rename 3c_GPT_Label_Certainty.ipynb => 2c_GPT_Label_Certainty.ipynb (100%)
Enumerating objects: 13, done.
Counting objects: 100% (13/13), done.
Delta compression using up to 8 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (7/7), 2.06 KiB | 2.06 MiB/s, done.
Total 7 (delta 4), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (4/4), completed with 4 local objects.
To github.com:T-Reiter/master-thesis.git
   856ec9b..09d0bda  master -> master


<a id="setup"></a>
## SANDBOX & SETUP

In [3]:
# Load API key from gitignored api_key module 
from utils.api_key import API_KEY as API_KEY

In [4]:
# base Request
# test capabilties
from openai import OpenAI

# Setup open AI client using api key
client = OpenAI(
  api_key=API_KEY
)

# Test prompt
return_number = "Return me a number between 1 and 100. Only return the number"

# Test completion
test_response = client.chat.completions.create(
  model="gpt-4o-mini",
  store=True,
  messages=[
    {"role": "user", 
     "content": return_number
    }
  ],
  temperature = 1 # default temp = 1  
)

print("Response: ", test_response.choices[0].message.content);

Response:  57


<a id="load_tweets"></a>
## Load, Partition, and Sample Tweets for Labeling

### Load the Tweets from the Kern et al. (2023) paper

In [5]:
# load the full tweets as used by CK
import pandas as pd
# pd.set_option('max_colwidth', 1000)
pd.set_option('display.max_colwidth', 0)

# Load the tweet files
folder_path = 'Tweets_CK/'
train_file = 'full_train_s.csv'
test_file = 'full_test_s.csv'

train_df = pd.read_csv(os.path.join(folder_path, train_file))
test_df = pd.read_csv(os.path.join(folder_path, test_file))

# Reorder test_df columns to match train_df
test_df = test_df[train_df.columns]

# Add the 'original_split' column
train_df['original_split'] = 'train'
test_df['original_split'] = 'test'

# Combine them
tweets_full = pd.concat([train_df, test_df], ignore_index=True)

# Rename columns for a consistent style 
tweets_full = (tweets_full.
               rename(columns={'tweet.id': 'tweet_id', 
                               'batch.tweet': 'batch_tweet',
                               'hate.speech': 'hate_speech', 
                               'offensive.language': 'offensive_language'}))

In [6]:
# Get Basic Info 
num_rows = tweets_full.shape[0]
num_cols = tweets_full.shape[1]
num_unique_tweets = tweets_full['tweet_id'].nunique()
tweet_counts = (tweets_full[['tweet_hashed','batch_tweet']]
                .drop_duplicates().groupby('batch_tweet').size())
num_batches = tweet_counts.mean()
average_tweets_per_batch = tweets_full['batch_tweet'].nunique()

print(f"Number of rows: {num_rows}")
print(f"Number of columns: {num_cols}")
print(f"Number of unique tweet IDs: {num_unique_tweets}")
print(f"Number of Batches: {num_batches}")
print(f'Average Tweets per Batch: {average_tweets_per_batch}')
tweets_full.sort_values('tweet_id').head()

Number of rows: 44900
Number of columns: 8
Number of unique tweet IDs: 3000
Number of Batches: 60.0
Average Tweets per Batch: 50


,tweet_id,id,version,batch_tweet,hate_speech,offensive_language,tweet_hashed,original_split
2927,1,7,B,R1,0.0,1.0,@###### bro that hoe live,train
2930,1,13,A,R1,0.0,1.0,@###### bro that hoe live,train
2932,1,155,A,R1,0.0,1.0,@###### bro that hoe live,train
2918,1,125,E,R1,1.0,0.0,@###### bro that hoe live,train
2919,1,173,E,R1,0.0,1.0,@###### bro that hoe live,train


### Partition the tweets into new Batches with IDs
In the original papers, the batch creation process does not seem to be included in the code.  
However, the tweets were randomly grouped into batches of 50.  
It is unclear whether this grouping was done based on the original tweet id gathered from elsewhere or whether the IDs were enumerated afterwards.  
In the end, we have have 50 tweets per batch (60 batches accordingly) that always appeared in the same order per batch.  
This randomized batch allocation process is replicated in the following.

In [7]:
# Randomly partition the df into batches reorder the tweets within batches 
BATCH_SIZE = 6

seed = 13
tweets_full_shuffled = (tweets_full[['tweet_id','tweet_hashed']]
                        .drop_duplicates()
                        .sample(frac=1, random_state=seed)
                        .reset_index(drop=True))
# create batch IDs
tweets_full_shuffled['batch_id'] = (tweets_full_shuffled.index // BATCH_SIZE) + 1
# create position IDs for each tweet within its batch
tweets_full_shuffled['tweet_in_batch'] = (tweets_full_shuffled.index % BATCH_SIZE) + 1

tweets_full_shuffled.head(5)

,tweet_id,tweet_hashed,batch_id,tweet_in_batch
0,811,Fat fucking funky nasty ass hoes,1,1
1,742,Up early then a bitch driving to denton omg can I move already,1,2
2,2098,@###### @###### there isn't a green one either. There's red and yellow in that pic...,1,3
3,2349,The only thing about niccas is they be followers smh .. Be ya self everyone else is taken!!,1,4
4,1371,@###### I'm the type to put that bitch in a river so you can gtf out my locations lol,1,5


### Clean tweets from escape characters etc.

In [8]:
# TO DO?

### Randomly Sample N tweets via batches and create the result df out of them

In [18]:
# sample tweets using the batch_ids 
# i.e., sample count = n_batch_samples * batch size 
# N_BATCH_SAMPLES = 20
N_BATCH_SAMPLES = 5
sampled_batches = (pd.Series(tweets_full_shuffled['batch_id']
                   .unique())
                   .sample(n=N_BATCH_SAMPLES
                           , random_state=13
                          ))

# get the tweets in these batches 
sampled_tweets = (tweets_full_shuffled[tweets_full_shuffled['batch_id']
                                       .isin(sampled_batches)]
                  [['batch_id','tweet_in_batch', 'tweet_id','tweet_hashed']]
                  .drop_duplicates())

print(f'Number of Batches Sampled: {N_BATCH_SAMPLES}')
print(f"Number of Unique Tweets Sampled: {len(sampled_tweets['tweet_hashed'])}")
print(f'Sampled Batch IDs: {sampled_batches.to_list()}')
sampled_tweets.iloc[0:3]

Number of Batches Sampled: 5
Number of Unique Tweets Sampled: 30
Sampled Batch IDs: [1, 335, 34, 265, 429]


,batch_id,tweet_in_batch,tweet_id,tweet_hashed
0,1,1,811,Fat fucking funky nasty ass hoes
1,1,2,742,Up early then a bitch driving to denton omg can I move already
2,1,3,2098,@###### @###### there isn't a green one either. There's red and yellow in that pic...


<a id="define_prompts"></a>
## Define the different Prompts

In [33]:
def_and_examples = """
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not hate speech. 
    For example,  
    “Why no boycott of the racist “redskins”? #Redskins @ChangeTheName”  
    This tweet contains a hateful term, but the tweeter is making an anti-racist statement. We would label tweet as containing offensive language.

    We define Offensive Language as:  
    Language that is highly offensive to certain individuals or groups but does not meet the requirements of hate speech. 

    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    This tweet contains offensive language, but it is not hate speech.   
    We would label it as offensive language.  

    Some tweets contains both hate speech AND offensive language.  
    “Subtweet me one more time, you dirty chink whore”  

    Some tweets do not meet our definition of hate speech or offensive language.  
    “Great lead battle and then Ricky hits Danica for a yellow. Oh boy. #NASCAR”  
"""



# Define the prompts
PROMPT_DICT = {
    # SINGLE TWEET PROMPTING
    ### A: Both OL and HS at the same time, OL first
    "A":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    {explanation}
    
    First, provide a label for OL, then provide a label for HS.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL, no
    
    Explanation (not part of the requested output):
    The first label 'OL' indicates that the tweet contains OL.
    The second label 'no' indicates that the tweet does not contain HS.
""", 
    ### B: Both OL and HS at the same time, HS first
    "B":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS)or/and Offensive Language (OL). 
    {explanation}
    
    First, provide a label for HS, then provide a label for OL.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    no, OL
    
    Explanation (not part of the requested output):
    The first label 'no' indicates that the tweet does not contain HS.
    The second label 'OL' indicates that the tweet contains OL.
    """, 
    ### C.OL: Only OL
    "C.OL":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    {explanation}
    
    Please provide a label for OL.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL
    
    Explanation (not part of the requested output):
    The label 'OL' indicates that the tweet contains OL.
""", 
    ### C.HS: Only HS
    "C.HS":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    {explanation}
    
    Please provide a label for HS.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    no
    
    Explanation (not part of the requested output):
    The label 'no' indicates that the tweet does not contain HS.
""",
    
    
    # SINGLE TWEET PROMPTING + CONFIDENCE SCORE
    ### A: Both OL and HS at the same time, OL first
    "A_conf":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    {explanation}
    
    For each tweet, provide EXACTLY TWO labels. 
    First, provide a label for OL, then provide a label for HS.
    For BOTH labels also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL 100, no 80
    
    Explanation (not part of the requested output):
    The first label 'OL 100' indicates that the tweet contains OL and that you are very confident about this label.
    The second label 'no' indicates that the tweet does not contain HS, and you are slightly less confident about this label.
""", 
    ### B: Both OL and HS at the same time, HS first
    "B_conf":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS) or/and Offensive Language (OL). 
    {explanation}
    
    For each tweet, provide EXACTLY TWO labels. 
    First, provide a label for HS, then provide a label for OL.
    For BOTH labels also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    no 80, OL 100
    
    Explanation (not part of the requested output):
    The first label 'no' indicates that the tweet does not contain HS, and you are slightly less confident about this label.
    The second label 'OL 100' indicates that the tweet contains OL and that you are very confident about this label.
""", 
    ### C.OL: Only OL
    "C.OL_conf":
    """You are given tweets and yor task is to tell whether they contain Offensive Language (OL). 
    {explanation}
    
    For each tweet, provide EXACTLY ONE label. 
    Please provide a label for OL.
    For the label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    OL 100
    
    Explanation (not part of the requested output):
    The label 'OL 100' indicates that the tweet contains OL and that you are very confident about this label.
""", 
    ### C.HS: Only HS
    "C.HS_conf":
    """You are given tweets and yor task is to tell whether they contain Hate Speech (HS). 
    {explanation}
    
    For each tweet, provide EXACTLY ONE label. 
    Please provide a label for HS.
    For the label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  

    Example Output:
    no 80
    
    Explanation (not part of the requested output):
    The label 'no' indicates that the tweet does not contain HS, and you are slightly less confident about this label.
""",
    
    
   

    # BATCH PROMPTING (of 50)
    ### B_A: batches with Both OL and HS at the same time (OL first)
    "A_batch":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    {explanation}
    
    For each tweet, provide a label for OL, then provide a label for HS.
    Separate the labels for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    OL, no; 
    OL, HS
    
    Explanation (not part of the requested output):
    Labels in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The first label in the first row 'yes' indicates that the first tweet contains OL.
    The second label in the first row 'no' indicates that the first tweet does not contain HS.
    The first label in the second row 'yes' indicates that the second tweet contains OL.
    The second label in the second row 'yes' indicates that the second tweet contains HS.
""", 
    ### B: Both OL and HS at the same time, HS first
    "B_batch":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Hate Speech (HS)or/and Offensive Language (OL) . 
    {explanation}
    
    For each tweet, provide a label for HS, then provide a label for OL.
    Separate the labels for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    no, yes; 
    yes, yes
    
    Explanation (not part of the requested output):
    Labels in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The first label in the first row 'no' indicates that the first tweet does not contain HS.
    The second label in the first row 'yes' indicates that the first tweet contains OL.
    The first label in the second row 'yes' indicates that the second tweet contains HS.
    The second label in the second row 'yes' indicates that the second tweet contains OL.
    
""", 
    ### C.OL: Only OL
    "C.OL_batch":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) . 
    {explanation}
    
    For each tweet, provide a label for OL.
    Separate the labels for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    yes; 
    yes
    
    Explanation (not part of the requested output):
    Labels in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The label in the first row 'yes' indicates that the first tweet contains OL.
    The label in the second row 'yes' indicates that the second tweet contains OL.
""", 
    ### C.HS: Only HS
    "C.HS_batch":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) . 
    {explanation}
    
    For each tweet, provide a label for OL.
    Separate the labels for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    no; 
    yes
    
    Explanation (not part of the requested output):
    Labels in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The label in the first row 'no' indicates that the first tweet does not contain HS.
    The label in the second row 'yes' indicates that the second tweet contains HS.
""",
    
    
    
    
    
    # BATCH PROMPTING WITH CONFIDENCE SCORES (of 50)
    ### B_A: batches with Both OL and HS at the same time (OL first)
    "A_batch_conf":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    {explanation}
    
    For each tweet, provide a label for OL, then provide a label for HS.
    For each label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.
    Separate the labels and confidence scores for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    yes 100, no 80; 
    yes 90, yes 90
    
    Explanation (not part of the requested output):
    Labels and scores in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The first label in the first row 'yes 100' indicates that the first tweet contains OL and that you are very confident about this label.
    The second label in the first row 'no 80' indicates that the first tweet does not contain HS and that you are slightly less confident about this label.
    The first label in the second row 'yes 90' indicates that the second tweet contains OL and that you are pretty confident about this label.
    The second label in the second row 'yes 90' indicates that the second tweet contains HS and that you are pretty confident about this label.
""", 
    
    ### B: Both OL and HS at the same time, HS first
    "B_batch_conf":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Hate Speech (HS) or/and Offensive Language (OL) . 
    {explanation}
    
    For each tweet, provide a label for HS, then provide a label for OL.
    For each label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.
    Separate the labels and confidence scores for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    no 80, yes 100; 
    yes 90, yes 90
    
    Explanation (not part of the requested output):
    Labels and scores in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The first label in the first row 'no 80' indicates that the first tweet does not contain HS and that you are slightly less confident about this label.
    The second label in the first row 'yes 100' indicates that the first tweet contains OL and that you are very confident about this label.
    The first label in the second row 'yes 90' indicates that the second tweet contains HS and that you are pretty confident about this label.
    The second label in the second row 'yes 90' indicates that the second tweet contains OL and that you are pretty confident about this label.
""", 
    ### C.OL: Only OL
    # Each tweet starts in a new line and is numbered with its index number and 3 closed brackets.
    "C.OL_batch_conf":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Offensive Language (OL) . 
    {explanation}
    
    For each tweet, provide a label for OL.
    For each label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.
    Separate the labels and confidence scores for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    yes 100; 
    yes 90
    
    Explanation (not part of the requested output):
    Labels and scores in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The label in the first row 'yes 100' indicates that the first tweet contains OL and that you are very confident about this label.
    The label in the second row 'yes 90' indicates that the second tweet contains OL and that you are pretty confident about this label.
""", 
    ### C.HS: Only HS
    "C.HS_batch_conf":
    """
    You are given multiple tweets and for each of them your task is to tell whether they contain Hate Speech (HS) . 
    {explanation}
    
    For each tweet, provide a label for HS.
    For each label also provide a score indicating your confidence in percent.
    0% represents lowest possible confidence, 100% represents highest possible confidence.
    Separate the labels and confidence scores for the different tweets with ; and start a new line

    Example Input: 
    “Guess who just got an apartment in downtown Columbus? That’s right bitch you guessed it, this guy.”  
    “Subtweet me one more time, you dirty chink whore”  

    Example Output:
    no 80; 
    yes 90
    
    Explanation (not part of the requested output):
    Labels and scores in the first row correspond to the first tweet, labels in the second row to the second tweet.
    The label in the first row 'no 80' indicates that the first tweet does not contain HS and that you are slightly less confident about this label.
    The label in the second row 'yes 90' indicates that the second tweet contains HS and that you are pretty confident about this label.
"""
}

<a id="run_labeling_single"></a>
## Run the Labeling Tasks (Single Tweet Prompting)

### Create Result DF

In [20]:
# SELECT CONDITIONS TO RUN THE LABELING FOR
# the selected tweets are stored in: sampled_tweets

# CONDITIONS  (choose out of the ones in the prompt dictionary)

CONDITIONS = ['A', 'B', 'C.HS', 'C.OL']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# # Create the DataFrame
results = pd.DataFrame(rows)
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
results.head()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label
0,1,1,811,Fat fucking funky nasty ass hoes,A,,,
1,1,1,811,Fat fucking funky nasty ass hoes,B,,,
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS,,,
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL,,,
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A,,,


In [21]:
row = 20

# Get Tweets and Conditions for the Loop
tweet_to_rate = results.iloc[row]['tweet']
condition = results.iloc[row]['condition']
condition = "A"
prompt_task = (PROMPT_DICT[condition]
          .format(explanation = def_and_examples))

print(prompt_task)
print(tweet_to_rate)

messages = [{"role": "system", "content": prompt_task},
               {"role": "user",   "content": tweet_to_rate}]

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages= messages,
    temperature=1,
    store=True,
)
label = resp.choices[0].message.content.strip()
print(label)

You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not 

In [22]:
%%time 

# Run the actual Loop for API Calls 

for row in range(0,results.shape[0]):
    
    # Get Tweets and Conditions for the Loop
    tweet_to_rate = results.iloc[row]['tweet']
    condition = results.iloc[row]['condition']
    prompt_task = (PROMPT_DICT[condition]
          .format(explanation = def_and_examples))
    
    # Define message
    messages = [{"role": "developer", "content": prompt_task},
                {"role": "user",   "content": tweet_to_rate}]
    
    # generate N_RESPONSES responses
    for response in range(1, N_RESPONSES + 1):
        # get response via API Call
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages= messages,
            temperature=1,
            store=True,
        )
        label = resp.choices[0].message.content.strip()
        # print(prompt)
        
        # store the result in the result df 
        # results.loc[[row], f"response_{response}"] = label
        results.at[row, f"R{response}_label"] = label
        
    # Print Progress
    print(f"""Finished Row {row+1} of {results.shape[0]} --> {round((row+1)/results.shape[0]*100,1)} %""", 
          end='\r')

n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results

Number of API Calls: 360--> 100.0 %
CPU times: user 4.04 s, sys: 278 ms, total: 4.32 s
Wall time: 3min 11s


,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label
0,1,1,811,Fat fucking funky nasty ass hoes,A,"yes, no","yes, no","yes, no"
1,1,1,811,Fat fucking funky nasty ass hoes,B,"yes, yes","no, yes","no, yes"
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS,yes,yes,yes
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL,yes,yes,yes
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A,"yes, no","yes, no","yes, no"
...,...,...,...,...,...,...,...,...
115,429,5,504,And when I say DOH DOH DOH DOH bitch that be Kdot,C.OL,yes,yes,yes
116,429,6,2724,RT @###### I saw a great chimpout 2 summers ago. A middle aged Salvadoran and his 15 Yr old son got wasted by 6 beaners at the metlife &#8230;,A,"yes, yes","yes, yes","yes, yes"
117,429,6,2724,RT @###### I saw a great chimpout 2 summers ago. A middle aged Salvadoran and his 15 Yr old son got wasted by 6 beaners at the metlife &#8230;,B,"yes, yes","yes, yes","yes, yes"
118,429,6,2724,RT @###### I saw a great chimpout 2 summers ago. A middle aged Salvadoran and his 15 Yr old son got wasted by 6 beaners at the metlife &#8230;,C.HS,yes,yes,yes


In [23]:
# save result file 
import os 
from datetime import date

tweet_count = (results['tweet'].drop_duplicates()
               .shape[0])
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/yes_no/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confno"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

<a id="run_labeling_single_conf"></a>
## Run the Labeling Task (with confidence scores)

### Create Result Df

In [27]:
# ISSUE: Loop is executed sequential s.t. we cannot use the acctual query-per-sec quote of the API (3r/s)
# SOL: --> rewrite code s.t. it's parallelized

# SELECT CONDITIONS TO RUN THE LABELING FOR
# the selected tweets are stored in: sampled_tweets

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_conf', 'B_conf', 'C.OL_conf', 'C.HS_conf']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3


rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# # Create the DataFrame
results = pd.DataFrame(rows)
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_score"] = ""    
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_Token1"] = ""   
    results[f"R{i}_Token2"] = ""  
    results[f"R{i}_Token3"] = ""     
    results[f"R{i}_Token4"] = ""     
    results[f"R{i}_Token5"] = ""     

    
results.head()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,...,R2_Token1,R2_Token2,R2_Token3,R2_Token4,R2_Token5,R3_Token1,R3_Token2,R3_Token3,R3_Token4,R3_Token5
0,1,1,811,Fat fucking funky nasty ass hoes,A_conf,,,,,,...,,,,,,,,,,
1,1,1,811,Fat fucking funky nasty ass hoes,B_conf,,,,,,...,,,,,,,,,,
2,1,1,811,Fat fucking funky nasty ass hoes,C.OL_conf,,,,,,...,,,,,,,,,,
3,1,1,811,Fat fucking funky nasty ass hoes,C.HS_conf,,,,,,...,,,,,,,,,,
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_conf,,,,,,...,,,,,,,,,,


### Run the Loop and score confidence scores

In [28]:
row = 1

# Get Tweets and Conditions for the Loop
tweet_to_rate = results.iloc[row]['tweet']
condition = results.iloc[row]['condition']
condition = "A_conf"
prompt_task = (PROMPT_DICT[condition]
          .format(explanation = def_and_examples))

print(prompt_task)
print(tweet_to_rate)

messages = [{"role": "system", "content": prompt_task},
               {"role": "user",   "content": tweet_to_rate}]

resp = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=1,
    store=True,
)
label = resp.choices[0].message.content.strip()
print(f'Response: {label}')

You are given tweets and yor task is to tell whether they contain Offensive Language (OL) or/and Hate Speech (HS). 
    
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms, but the tweet is not 

In [42]:
import re
resp
labels = []
labels.extend(resp.findall(r'\b(yes|no)\b', s, flags=re.IGNORECASE))
labels

AttributeError: 'str' object has no attribute 'findall'

In [38]:
# %%time 

# Run the actual Loop for API Calls 
N_TOP_LOGPROBS = 5

for row in range(0,results.shape[0]):
    
    # Get Tweets and Conditions for the Loop
    tweet_to_rate = results.iloc[row]['tweet']
    condition = results.iloc[row]['condition']
    prompt_task = (PROMPT_DICT[condition]
               .format(explanation = def_and_examples))
    
    # Define message
    messages = [{"role": "developer", "content": prompt_task},
                {"role": "user",   "content": tweet_to_rate}]
    
    # generate N_RESPONSES responses
    for response in range(1, N_RESPONSES + 1):
        # get response via API Call
        resp_full = client.chat.completions.create(
            model="gpt-4o-mini",
            messages= messages,
            temperature=1,
            logprobs=True,
            top_logprobs= N_TOP_LOGPROBS,
            store=True,
        )
        # Extract the Labels & Confidence Score
        resp = resp_full.choices[0].message.content.strip()
        print(condition)
        print(resp)
        resp_split = resp.split(',')
        label = resp_split[0]
        score = resp_split[1].strip()
        
        # Extract the Token Logprobs for tokens 1 to 5
        # exract logprobs for first token
        top_k_logprobs_first = resp_full.choices[0].logprobs.content[0].top_logprobs
        top_n_tokens_first = [obj.token for obj in top_k_logprobs_first[:N_TOP_LOGPROBS]]
        top_n_logprobs_first = [obj.logprob for obj in top_k_logprobs_first[:N_TOP_LOGPROBS]]
        # token_value_dict = dict(zip(top_n_tokens_first, top_n_logprobs_first))
        token_value_series_first = pd.Series(top_n_logprobs_first, index=top_n_tokens_first)

        # exract logprobs for second token
        top_k_logprobs_second = resp_full.choices[0].logprobs.content[1].top_logprobs
        top_n_tokens_second = [obj.token for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
        top_n_logprobs_second = [obj.logprob for obj in top_k_logprobs_second[:N_TOP_LOGPROBS]]
        token_value_series_second = pd.Series(top_n_logprobs_second, index=top_n_tokens_second)

        # exract logprobs for third token
        top_k_logprobs_third = resp_full.choices[0].logprobs.content[2].top_logprobs
        top_n_tokens_third = [obj.token for obj in top_k_logprobs_third[:N_TOP_LOGPROBS]]
        top_n_logprobs_third = [obj.logprob for obj in top_k_logprobs_third[:N_TOP_LOGPROBS]]
        token_value_series_third = pd.Series(top_n_logprobs_third, index=top_n_tokens_third)
        
        # exract logprobs for fourth token
        top_k_logprobs_fourth = resp_full.choices[0].logprobs.content[3].top_logprobs
        top_n_tokens_fourth = [obj.token for obj in top_k_logprobs_fourth[:N_TOP_LOGPROBS]]
        top_n_logprobs_fourth = [obj.logprob for obj in top_k_logprobs_fourth[:N_TOP_LOGPROBS]]
        token_value_series_fourth = pd.Series(top_n_logprobs_fourth, index=top_n_tokens_fourth)
        
        # exract logprobs for fifth token
        top_k_logprobs_fifth = resp_full.choices[0].logprobs.content[4].top_logprobs
        top_n_tokens_fifth = [obj.token for obj in top_k_logprobs_fifth[:N_TOP_LOGPROBS]]
        top_n_logprobs_fifth = [obj.logprob for obj in top_k_logprobs_fifth[:N_TOP_LOGPROBS]]
        token_value_series_fifth = pd.Series(top_n_logprobs_fifth, index=top_n_tokens_fifth)

        
        # store the result in the result df 
        # results.loc[[row], f"response_{response}"] = label
        results.at[row, f"R{response}_label"] = label
        results.at[row, f"R{response}_score"] = score
        results.at[row, f"R{response}_Token1"] = token_value_series_first
        results.at[row, f"R{response}_Token2"] = token_value_series_second
        results.at[row, f"R{response}_Token3"] = token_value_series_third
        results.at[row, f"R{response}_Token4"] = token_value_series_fourth
        results.at[row, f"R{response}_Token5"] = token_value_series_fifth
        
    # Print Progress
    # print(f"""Finished Row {row+1} of {results.shape[0]} --> {round((row+1)/results.shape[0]*100,1)} %""", end='\r')


n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(4)

A_conf
yes 95, no 85
A_conf
yes 90, no 85
A_conf
yes 90, no 80
B_conf
yes 90, yes 95
B_conf
yes 90, yes 100
B_conf
yes 90, yes 90
C.OL_conf
yes 95


IndexError: list index out of range

In [123]:
# save result file 
import os 
from datetime import date

tweet_count = (results['tweet'].drop_duplicates()
               .shape[0])
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confyes"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

<a id="run_labeling_batch"></a>
## Run the Labeling Tasks (Batch Prompting)

### Create Results DF

In [39]:
# SELECT CONDITIONS TO RUN THE LABELING FOR

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_batch', 'B_batch', 'C.HS_batch', 'C.OL_batch']
# CONDITIONS = ['A', 'B', 'C.HS', 'C.OL', 
#               'A_ex', 'B_ex', 'C_ex.HS', 'C_ex.OL'] 

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# Create the DataFrame
results = pd.DataFrame(rows)
results = results.sort_values(['batch_id', 'tweet_in_batch', 'condition'])

# Add response columns
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""

results.head()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label
0,1,1,811,Fat fucking funky nasty ass hoes,A_batch,,,
1,1,1,811,Fat fucking funky nasty ass hoes,B_batch,,,
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS_batch,,,
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL_batch,,,
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_batch,,,


### Run the Loop

In [40]:
%%time 

# Run the actual Loop for API Calls 
# now we have too loop through conditions first and batches second
# now we have to loop through batches and provide all tweets in the prompt 
# then we will get different responses 

conditions = results['condition'].unique()
batches = results['batch_id'].unique()

N_RESPONSES = 3

# one condition after the other
for condition in conditions:
    # print(condition)
    
    # loop through batches in the conditions
    for batchID in batches:
        
        
        # get the tweets within each batch and create the prompt 
        tweets_in_batch = (results[results['batch_id'] == batchID]
                       ['tweet']
                       .unique())
        tweet_indices = ((results['batch_id'] == batchID) & 
                         (results['condition'] == condition))
        tweets_in_batch_str = "\n" + "\n".join(
            [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
        )
        prompt_task = (PROMPT_DICT[condition]
                   .format(explanation = def_and_examples))
        
        
        # Define message
        messages = [{"role": "developer", "content": prompt_task},
                    {"role": "user",   "content": tweets_in_batch_str}]
        
        
        
        
        # generate N_RESPONSES responses
        for response in range(1, N_RESPONSES + 1):
            
            valid_response = False
            
            while not valid_response:
                
            
                # get response via API Call
                resp = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages= messages,
                    temperature=1,
                    store=True,
                )
                labels = resp.choices[0].message.content.strip()
                labels_list = labels.split(';')
                labels_list
                
                # Check if we have the desired format 
                # (6 responses with each having a label and a confidence score
                if len(labels_list) == 6:
                    valid_response = True
                else:
                    continue
                

            # Store the different labels in the correct df rows 
            ### Get the result df indices for the tweets in the current batch
            subset_indices = results.loc[tweet_indices].index.to_list()
            # store results
            for i, idx in enumerate(subset_indices):
                results.at[idx, f"R{response}_label"] = labels_list[i]
    
        
#         # generate N_RESPONSES responses
#         for response in range(1, N_RESPONSES + 1):
#             # get response via API Call
#             resp = client.chat.completions.create(
#                 model="gpt-4o-mini",
#                 messages=messages,
#                 temperature=1,
#                 store=True,
#             )
#             labels = resp.choices[0].message.content.strip()
#             labels_list = labels.split(';')
#             labels_list

        
#             # Store the different labels in the correct df rows 
#             ### Get the result df indices for the tweets in the current batch
#             subset_indices = results.loc[tweet_indices].index.to_list()
#             # store results
#             for i, idx in enumerate(subset_indices):
#                 results.at[idx, f"R{response}_label"] = labels_list[i]
                

        # Print Progress
        message = f"""Finished Batch {batches.tolist().index(batchID) + 1} in Condition {conditions.tolist().index(condition) + 1} --> \
        {((conditions.tolist().index(condition)) * len(batches) + (batches.tolist().index(batchID) + 1)) / (len(conditions) * len(batches)) * 100:.2f} %"""
        print('\r' + message.ljust(100), end='')
        sys.stdout.flush()        
       
    
n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(5)

Finished Batch 30 in Condition 4 -->         100.00 %                                               Number of API Calls: 2160
CPU times: user 4.94 s, sys: 448 ms, total: 5.39 s
Wall time: 3min 57s


,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label
0,1,1,811,Fat fucking funky nasty ass hoes,A_batch,OL,OL,OL
1,1,1,811,Fat fucking funky nasty ass hoes,B_batch,OL,OL,OL
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS_batch,none,none,none
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL_batch,OL,OL,OL
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_batch,OL,OL,OL


In [41]:
# save result file 
import os 
from datetime import date

tweet_count = (results['tweet'].drop_duplicates()
               .shape[0])
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confno_batch"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

<a id="run_labeling_batch_confidence"></a>
## Run the Labeling Tasks (Batch Prompting) + Self-Report Confidence Scores

### Create Result Df (use the sample sampled batches as before)

In [42]:
# SELECT CONDITIONS TO RUN THE LABELING FOR

# CONDITIONS  (choose out of: A, B, C.HS, C.OL)

CONDITIONS = ['A_batch_conf', 'B_batch_conf', 
              'C.HS_batch_conf', 'C.OL_batch_conf']

# Loop through each row and call the API N_RESPONSES times
N_RESPONSES = 3

rows = []
for idx, row in sampled_tweets.iterrows():
    batch_id = row["batch_id"]
    tweet_in_batch = row["tweet_in_batch"]
    original_tweet_id = row["tweet_id"]
    tweet_text = row["tweet_hashed"]
    for condition in CONDITIONS:
        rows.append({
            "batch_id": batch_id,
            "tweet_in_batch": tweet_in_batch,
            "tweet_id": original_tweet_id,      
            "tweet": tweet_text,
            "condition": condition
        })

# Create the DataFrame
results = pd.DataFrame(rows)
results = results.sort_values(['batch_id', 'tweet_in_batch', 'condition'])

# Add response columns
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_label"] = ""
    
for i in range(1, N_RESPONSES + 1):
    results[f"R{i}_score"] = ""    

results.tail()

,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score
715,486,5,645,RT @###### &#8220;@###### &#8220;@###### okay now I'm going to Krispy Kreme for real&#8221;fat bitch&#8221;@###### this nigga trying &#8230;,C.OL_batch_conf,,,,,,
716,486,6,734,"Look at this little faggot.""@###### Tell her she's beautiful as much as you can. It'll make her smile""",A_batch_conf,,,,,,
717,486,6,734,"Look at this little faggot.""@###### Tell her she's beautiful as much as you can. It'll make her smile""",B_batch_conf,,,,,,
718,486,6,734,"Look at this little faggot.""@###### Tell her she's beautiful as much as you can. It'll make her smile""",C.HS_batch_conf,,,,,,
719,486,6,734,"Look at this little faggot.""@###### Tell her she's beautiful as much as you can. It'll make her smile""",C.OL_batch_conf,,,,,,


### Run the Loop

In [43]:
# testrun to check output format

batchID = 486
condition = 'A_batch_conf'

# batchID = results['batch_id'][0]
# condition = results['condition'][0]
tweets_in_batch = (results[results['batch_id'] == batchID]
                   ['tweet']
                   .unique())
tweets_in_batch_str = "\n" + "\n".join(
    [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
)

tweets_in_batch_str_test = 'Before each label, return the index number of the respective tweet.' + tweets_in_batch_str


prompt_task = (PROMPT_DICT[condition]
                   .format(explanation = def_and_examples))
        
        
# Define message
messages = [{"role": "developer", "content": prompt_task},
            {"role": "user",   "content": tweets_in_batch_str}]

resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=1,
            store=True,
        )

response = resp.choices[0].message.content.strip()
responses = response.split(';')
responses

print(prompt_task)
print(tweets_in_batch_str)

print(f"\n Response: ", response)
# labels = resp.choices[0].message.content.strip()
# labels_list = labels.split(';')
# labels_list

You are given multiple tweets and for each of them your task is to tell whether it contains Offensive Language (OL) or Hate Speech (HS). 
    
    We define hate speech as:  
    Language that is used to expresses hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.

    Here are examples of tweets that contain hate speech, according to our definition:  
    "You are a jiggaboo...!”  
    “they’re working on a bill to prevent retards from voting. who knew retards COULD vote? things are starting to make sense now.”  
    “Every slant in #LA should be deported. Those scum have no right to be here. Chinatown should be bulldozed,”

    Remember, hate speech is: language that is used to express hatred towards a targeted group or language intended to be derogatory, to humiliate, or to insult the members of the group.  
    However, context matters. If a tweet discusses someone else’s hate speech, it may use hateful terms,

In [44]:
# %%time 

# Run the actual Loop for API Calls 
# now we have too loop through conditions first and batches second
# now we have to loop through batches and provide all tweets in the prompt 
# then we will get different responses 

conditions = results['condition'].unique()
batches = results['batch_id'].unique()

N_RESPONSES = 3

# one condition after the other
for condition in conditions:
    # print(condition)
    
    # loop through batches in the conditions
    for batchID in batches:
        
        # get the tweets within each batch and create the prompt 
        tweets_in_batch = (results[results['batch_id'] == batchID]
                       ['tweet']
                       .unique())
        tweet_indices = ((results['batch_id'] == batchID) & 
                         (results['condition'] == condition))
        tweets_in_batch_str = "\n" + "\n".join(
            [f"{i+1}) {tweet}" for i, tweet in enumerate(tweets_in_batch)]
        )
        prompt_task = (PROMPT_DICT[condition]
                   .format(explanation = def_and_examples))
        
        messages = [{"role": "developer", "content": prompt_task},
                    {"role": "user",   "content": tweets_in_batch_str}]

        
        # generate N_RESPONSES responses
        for response in range(1, N_RESPONSES + 1):
            
            valid_response = False
            
            while not valid_response:
                
            
                # get response via API Call
                resp = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages= messages,
                    temperature=1,
                    store=True,
                )
                api_response_full = resp.choices[0].message.content.strip()
                api_responses = api_response_full.split(';')
                api_responses_per_tweet = [api_response.split(',') for api_response in api_responses]
                
                # Check if we have the desired format 
                # (6 responses with each having a label and a confidence score
                if len(api_responses_per_tweet) == 6:
                    valid_response = True
                else:
                    continue
                    

            # Extract the Labels & Confidence Scores per tweet
            labels_list = [tweet_response[0].strip() 
                           for tweet_response in api_responses_per_tweet]
            scores_list = [tweet_response[1].strip() 
                           for tweet_response in api_responses_per_tweet]


            # Store the different labels in the correct df rows 
            ### Get the result df indices for the tweets in the current batch
            subset_indices = results.loc[tweet_indices].index.to_list()
            # store results
            for i, idx in enumerate(subset_indices):
                results.at[idx, f"R{response}_label"] = labels_list[i]
                results.at[idx, f"R{response}_score"] = scores_list[i]
                
        # Print Progress
        message = f"""Finished Batch {batches.tolist().index(batchID) + 1} in Condition {conditions.tolist().index(condition) + 1} --> \
        {((conditions.tolist().index(condition)) * len(batches) + (batches.tolist().index(batchID) + 1)) / (len(conditions) * len(batches)) * 100:.2f} %"""
        print('\r' + message.ljust(100), end='')
        sys.stdout.flush()    
       
    
n_prompts = results.shape[0]*N_RESPONSES
print(f"Number of API Calls: {n_prompts}")
results.head(5)

Finished Batch 30 in Condition 4 -->         100.00 %                                               Number of API Calls: 2160


,batch_id,tweet_in_batch,tweet_id,tweet,condition,R1_label,R2_label,R3_label,R1_score,R2_score,R3_score
0,1,1,811,Fat fucking funky nasty ass hoes,A_batch_conf,OLHS,OLHS,OLHS,4,4,4
1,1,1,811,Fat fucking funky nasty ass hoes,B_batch_conf,OL,OL,OL,3,3,3
2,1,1,811,Fat fucking funky nasty ass hoes,C.HS_batch_conf,none,none,none,4,2,4
3,1,1,811,Fat fucking funky nasty ass hoes,C.OL_batch_conf,OL,OL,OL,4,4,4
4,1,2,742,Up early then a bitch driving to denton omg can I move already,A_batch_conf,OL,OL,OL,3,3,3


In [45]:
# save result file 
import os 
from datetime import date

tweet_count = (results['tweet'].drop_duplicates()
               .shape[0])
condition_count = (results['condition'].drop_duplicates()
                   .shape[0])
response_count = (results.loc[:, results.columns.str.endswith('_label')]
                  .shape[1])

output_dir = "Data_Collection/"
today = date.today().strftime("%Y_%m_%d")
suffix = f"explanation_{tweet_count}t_{condition_count}c_{response_count}r_confyes_batch"
filename = f"{today}_gpt_labels_{suffix}.csv"

filepath = os.path.join(output_dir, filename)
results.to_csv(filepath, index=False)

# Notes 
1) The tweets can be processed in parts to either (e.g., the first 2k for all conditions)  
A) be able to split the task into free sub parts or   
B) to be able to better estimate the total costs  

2) Probably the Batch Calls have to be done in a separate code snippet

3) Costs: 
4o-mini : 
Input:          15ct  /1M Tokens
Cached Input:   7.5ct /1M Tokens (might be our case)
Output:         60ct  /1M Tokens


In [39]:
# Get Token Counts 
from utils import count_tokens
import statistics
# count tokens for gpt-3.5-turbo model (default in tiktoken package 
# print(PROMPT_DICT['A'])
# print(sampled_tweets['tweet_hashed'])
print(count_tokens(PROMPT_DICT['A'])) # 111 TOKENS for standard condition prompt 
print(statistics.fmean([count_tokens(x) for x in sampled_tweets['tweet_hashed']])) # 15 Average Tweet Token length
# response has max 2 tokens

121
21.94
